# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [17]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [18]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 3, 20)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 15)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [19]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 3 market file(s)
Unique markets (all):  303,101
Unique markets (filtered 2026-01-01 → 2026-03-20): 293,807


,end_date_iso,condition_id,market_json
0,2026-01-01T00:00:00Z,0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-10-20T21:13:17Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad"",""question_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06825"",""question"":""Will Rasmr win the Synthetix trading competition?"",""description"":""This market will resolve to the listed participant with the highest PnL at the end of the Synthetix trading competition starting on October 20, 2025.\n\nThe resolution source will be the Synthetix Leaderboard on https://www.synthetix.io/.\n\nIf two or more participants are tied for the highest PnL at the time of resolution, the market will resolve to the participant whose display name appears first in alphabetical order on the official Synthetix leaderboard.\n\nIf the competition is not completed by December 31, 2025, 11:59 PM ET, or if the official leaderboard or results are unavailable, the market will remain open until reliable data is published by Synthetix. If no such data is available by December 31, 2025, the market will resolve to “Other.”\n\nNote: The official competition includes 100 participants, but this market will resolve solely based on a selected group of KOL participants. Traders who are not eligible KOLs will not be considered. E.g. an eligible KOL will be considered to have won if they are the top trader out of eligible KOLs regardless of if one of the non KOL participants finishes ahead of them. \n\nAdditional KOLs may be added as options later to reflect the full set of eligible KOLs."",""market_slug"":""will-rasmr-win-the-synthetix-trading-competition-933"",""end_date_iso"":""2026-01-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":true,""neg_risk_market_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06800"",""neg_risk_request_id"":""0xda26edfbf8e42092e0693c8f4cedd5319c371ff9bd8650f2d7d7b00b19ce6fa2"",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""rewards"":{""rates"":null,""min_size"":50,""max_spread"":3.5},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""67487832607016995738972990163663499867372390570812445168484453192396152205498"",""outcome"":""Yes"",""price"":0,""winner"":false},{""token_id"":""99393257971182825969115164966386446587421508634202783327226489236020763806664"",""outcome"":""No"",""price"":1,""winner"":true}],""tags"":[""Crypto""]}"
1,2026-01-01T00:00:00Z,0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-12-31T14:01:53Z"",""minimum_order_size"":5,""minimum_tick_size"":0.01,""condition_id"":""0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9"",""question_id"":""0x9c4aa152472d5b3332f1760add02bb13dc6804e1fe4021e35d8a3ecd8d1a4e29"",""question"":""Bitcoin Up or Down - January 1, 9:00AM-9:15AM ET"",""description"":""This market will resolve to \""Up\"" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to \""Down\"".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/

In [20]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 583,607
Market meta rows:     293,807


## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [21]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.shard_processor import (
    select_top_wallets_shard,
    enrich_and_group_shard,
)

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade shard(s)")

sample_raw = pd.read_parquet(trade_files[0])
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)

Found 16 trade shard(s)
Sample shard rows (0.parquet): 5,498,880
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,78753205165658130534456351077321496563862891920229742523513427553266682271361,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",SELL,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38fce30001acc0181bf112,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,Yes,58665055277986534416719939405914621458010137331379938342097742389618466217100,0.37,11.1


In [22]:
# ── build token resolution DataFrame ────────────────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnl: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS): f
        for f in trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnl or pnl > shard_wallet_pnl[w]:
                shard_wallet_pnl[w] = pnl
        if i % 4 == 0 or i == len(trade_files):
            print(
                f"  {i}/{len(trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnl):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnl.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")

Phase 1 — selecting top-4% wallets per shard...
  4/16 shards | raw: 22,971,278 | in-range: 14,056,406 | wallet union so far: 11,936
  8/16 shards | raw: 46,799,103 | in-range: 28,682,743 | wallet union so far: 19,318
  12/16 shards | raw: 69,523,558 | in-range: 42,543,920 | wallet union so far: 24,112
  16/16 shards | raw: 93,357,019 | in-range: 57,193,947 | wallet union so far: 28,507

Phase 1 done. Candidate wallets (union of top-4% per shard): 28,507


## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [23]:
# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(trade_files):
            print(f"  {i}/{len(trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 27,370,136
  Unique wallets in grouped:                   28,507


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,avg_price
0,0xb16ce0318e26bccd2b3fa98572a1f3e953941319a65ea2f1338c79d15210413f,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.0,12.285713,12.285713,5.405714,12.285713,1,6.879999,0.44
1,0xa43e7f0f8c8d765da347395ebe97f9d8aaa34df3e0d079e2e0704c255003013c,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.0,21.196426,8.910713,3.920714,8.910713,1,4.989999,0.44
2,0x3604ec14512d8ee691ad062768743212fdc70245ce5998cb26da7df8feeca8b2,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.0,49.624996,28.428570,12.508571,28.428570,1,15.919999,0.44
3,0xe7300a94643440ab691becc1122b0816b8944d99b6f7ed2a2036e6fcbd1f9125,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.0,99.984996,50.360000,22.158400,50.360000,1,28.201600,0.44
4,0xd89bb82e1dd92941fc3c9cc1636b2f6582bd21508f1195c6d1e76df9f5a54edc,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:19:10+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.0,784.600379,684.615383,355.999999,684.615383,2,328.615384,0.52


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [24]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        pnl_usdc        = ("trade_pnl",        "sum"),
    )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
shard_pnl_values = list(shard_wallet_pnl.values())
pnl_cutoff = statistics.median(shard_pnl_values)

final_wallets = set(
    wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_cutoff, "wallet"]
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)

Unique wallets after Phase 2 union:    28,507
Per-shard PnL median (cut-off):        1,086.61 USDC
Wallets passing PnL cut-off:           10,964


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,663,52672,4.946852e+07,6.970130e+06
1,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,319,50996,2.666777e+07,2.796902e+06
2,0x876426b52898c295848f56760dd24b55eda2604a,76,10113,3.927546e+06,1.554148e+06
3,0x006cc834cc092684f1b56626e23bedb3835c16ea,162,1420,1.431709e+07,1.493206e+06
4,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,12769,430092,3.394417e+07,1.465410e+06


## 5 · Market-level volume summary

In [25]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 137,186


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,2089,192266,1.276037e+08,1.319972e+08
1,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28 00:00:00+00:00,1879,116753,8.534282e+07,9.108129e+07
2,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31 00:00:00+00:00,2128,133451,5.529724e+07,5.933645e+07
3,0xa8b744720006da3c08b4dc8a61a5ce930542f550fcf8d27380ae898de636799d,Khamenei out as Supreme Leader of Iran by January 31?,2026-01-31 00:00:00+00:00,1743,89226,3.879687e+07,4.039246e+07
4,0xabb86b080e9858dcb3f46954010e49b6f539c20036856c7f999395bfd58d01e6,"US strikes Iran by January 31, 2026?",2026-01-31 00:00:00+00:00,2153,143532,3.213394e+07,3.367401e+07
5,0x204d24f3a0f5dd5fca825292bdeab6a97af3978b2caa2b21bb37e610eddfff5d,Seahawks vs. Patriots,2026-02-08 00:00:00+00:00,1751,78280,3.201824e+07,3.207574e+07
6,0xb3dbb0f2e1df15cc820e0cde749eb112bf160994ec53a8ac57dceff9dbe5007e,"Will Melania say ""Career"" during AI talk on Friday?",2026-01-16 00:00:00+00:00,189,7259,2.681767e+07,2.684749e+07
7,0x05beef793157deb1cc34e2d3159539f461b73c7eeaa3b5348617c10eed5509d0,U.S. anti-cartel ground operation in Mexico by January 31?,2026-01-31 00:00:00+00:00,445,17809,2.661320e+07,2.678493e+07
8,0x41e47408f8ab39b46a9d9e3c9b15ebd62f1d795eb072ff46df3d376c09eb583e,"US x Iran meeting by February 6, 2026?",2026-02-13 00:00:00+00:00,417,20226,1.882669e+07,1.897612e+07
9,0xb8c1bd306a8a4cedfb280e114e655c5092b3f37edccae05cd877d7f21a5774ce,"Russia x Ukraine ceasefire by January 31, 2026?",2026-01-31 00:00:00+00:00,764,35981,1.688970e+07,1.705663e+07


## 6 · Wallet statistics (quantiles)

In [26]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]    = wallet_stats["pnl_usdc"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,29,1.0,1.0,-395600.65,28478
0.01,285,1.0,1.0,-37040.25,28222
0.05,1425,2.0,1.0,-6836.19,27082
0.1,2851,2.0,1.0,-2806.62,25656
0.25,7127,6.0,2.0,-614.04,21380
0.5,14254,28.0,10.0,687.95,14253
0.75,21380,165.0,22.0,1830.00,7127
0.9,25656,847.0,112.0,4035.20,2851
0.95,27082,2093.0,247.0,8733.59,1425


## 7 · Full enriched trade table

In [27]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0xb16ce0318e26bccd2b3fa98572a1f3e953941319a65ea2f1338c79d15210413f,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,12.285713,12.285713,0.440,True,1.0,5.405714,12.285713,6.879999,1
1,0xa43e7f0f8c8d765da347395ebe97f9d8aaa34df3e0d079e2e0704c255003013c,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,21.196426,8.910713,0.440,True,1.0,3.920714,8.910713,4.989999,1
2,0x3604ec14512d8ee691ad062768743212fdc70245ce5998cb26da7df8feeca8b2,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,49.624996,28.428570,0.440,True,1.0,12.508571,28.428570,15.919999,1
3,0xe7300a94643440ab691becc1122b0816b8944d99b6f7ed2a2036e6fcbd1f9125,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,99.984996,50.360000,0.440,True,1.0,22.158400,50.360000,28.201600,1
4,0xd89bb82e1dd92941fc3c9cc1636b2f6582bd21508f1195c6d1e76df9f5a54edc,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:19:10+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,784.600379,684.615383,0.520,True,1.0,355.999999,684.615383,328.615384,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27370131,0x3bcd973a3ee5617ac8b063536a42ebc524c9863ceaab5833c57501df536833ca,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-08 18:51:25+00:00,0xc734f61dbb7fbf5ff7afac8c18b1f05f13c4b9b408f8f8d6d210a862ebabd00d,Yes,2500.000000,1000.000000,0.006,False,0.0,6.000000,0.000000,-6.000000,1
27370132,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c5e19be1c3ea3b211eddd,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,200.000000,200.000000,0.020,False,0.0,4.000000,0.000000,-4.000000,1
27370133,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5e2ff1c46933b30c74547,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,4200.000000,4000.000000,0.003,False,0.0,12.000000,0.000000,-12.000000,1
27370134,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7c859f3fc1bd1ce740b6f,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Yes,530.000000,50.000000,0.507,True,1.0,25.350000,50.000000,-24.650000,2


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [28]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 10,964


In [29]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  14,908,057
  training rows: 10,022,715
  test rows:     4,885,342
Output shards:  16
  0.parquet  (869,085 rows)
  1.parquet  (883,948 rows)
  2.parquet  (963,782 rows)
  3.parquet  (933,157 rows)
  4.parquet  (1,023,569 rows)
  5.parquet  (892,489 rows)
  6.parquet  (992,580 rows)
  7.parquet  (960,343 rows)
  8.parquet  (878,007 rows)
  9.parquet  (891,158 rows)
  a.parquet  (1,017,347 rows)
  b.parquet  (922,739 rows)
  c.parquet  (875,302 rows)
  d.parquet  (980,102 rows)
  e.parquet  (908,135 rows)
  f.parquet  (916,314 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [30]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.

In [31]:
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")

INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 10,964 entries
